In [13]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [14]:
import json

from src.config import RAW_DOCUMENTS_DIR, TOP_K
from src.loader import load_documents
from src.pipeline import build_knowledge_base
from src.retriever import retrieve

In [15]:
import json
from pathlib import Path

project_root = Path.cwd().parent

evaluation_file = (
    project_root / "evaluation" / "questions.json"
)

with open(
    evaluation_file,
    "r",
    encoding="utf-8",
) as file:
    questions = json.load(file)

len(questions)

4

In [16]:
documents = load_documents(
    RAW_DOCUMENTS_DIR
)

model, index, chunks = build_knowledge_base(
    documents
)

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1991.00it/s]


Documents: 1
Chunks: 1


In [17]:
question = questions[0]["question"]

results = retrieve(
    query=question,
    model=model,
    index=index,
    chunks=chunks,
    top_k=3,
)

for result in results:
    print(f"Score: {result['score']:.3f}")
    print(result["text"])
    print("-" * 60)

Score: 0.728
Company Annual Leave Policy Employees receive 30 days of annual leave per calendar year. Employees should request annual leave through the company's HR system. Unused annual leave may be carried over to the following year, subject to company policy. Employees should normally request leave at least two weeks in advance. Remote Work Policy Employees may work remotely up to three days per week with manager approval. Employees working remotely must remain available during their agreed working hours.
------------------------------------------------------------


In [18]:
expected_text = questions[0]["expected_text"]

retrieved_text = " ".join(
    result["text"]
    for result in results
)

expected_text.lower() in retrieved_text.lower()

True

In [19]:
results_summary = []

for item in questions:

    results = retrieve(
        query=item["question"],
        model=model,
        index=index,
        chunks=chunks,
        top_k=TOP_K,
    )

    retrieved_text = " ".join(
        result["text"]
        for result in results
    )

    success = (
        item["expected_text"].lower()
        in retrieved_text.lower()
    )

    results_summary.append(
        {
            "question": item["question"],
            "success": success,
        }
    )

results_summary

[{'question': 'How many days of annual leave do employees receive?',
  'success': True},
 {'question': 'How far in advance should employees normally request leave?',
  'success': True},
 {'question': 'How many days per week can employees work remotely?',
  'success': True},
 {'question': 'What must remote employees do during their working hours?',
  'success': True}]

In [20]:
successful = sum(
    result["success"]
    for result in results_summary
)

total = len(results_summary)

recall_at_3 = successful / total

print(f"Recall@3: {recall_at_3:.2%}")

Recall@3: 100.00%


In [21]:
k_values = [1, 3, 5]

for k in k_values:

    successful = 0

    for item in questions:

        results = retrieve(
            query=item["question"],
            model=model,
            index=index,
            chunks=chunks,
            top_k=k,
        )

        retrieved_text = " ".join(
            result["text"]
            for result in results
        )

        if item["expected_text"].lower() in retrieved_text.lower():
            successful += 1

    recall = successful / len(questions)

    print(f"Recall@{k}: {recall:.2%}")

Recall@1: 100.00%
Recall@3: 100.00%
Recall@5: 100.00%


In [22]:
documents = load_documents(
    RAW_DOCUMENTS_DIR
)

for document in documents:
    print(document)

{'text': "Company Annual Leave Policy\n\nEmployees receive 30 days of annual leave per calendar year.\n\nEmployees should request annual leave through the company's HR system.\n\nUnused annual leave may be carried over to the following year, subject to company policy.\n\nEmployees should normally request leave at least two weeks in advance.\n\nRemote Work Policy\n\nEmployees may work remotely up to three days per week with manager approval.\n\nEmployees working remotely must remain available during their agreed working hours.", 'source': 'company_policy.txt', 'page': None}


In [23]:
def reciprocal_rank(
    retrieved_results,
    expected_text,
):
    """
    Return the reciprocal rank of the first
    relevant retrieved chunk.

    Example:
    rank 1 -> 1.0
    rank 2 -> 0.5
    rank 3 -> 0.333
    no result -> 0.0
    """

    for rank, result in enumerate(
        retrieved_results,
        start=1,
    ):

        if (
            expected_text.lower()
            in result["text"].lower()
        ):
            return 1 / rank

    return 0.0

In [24]:
item = questions[0]

results = retrieve(
    query=item["question"],
    model=model,
    index=index,
    chunks=chunks,
    top_k=3,
)

score = reciprocal_rank(
    results,
    item["expected_text"],
)

print(f"Reciprocal Rank: {score:.3f}")

Reciprocal Rank: 1.000


In [25]:
reciprocal_ranks = []

for item in questions:

    results = retrieve(
        query=item["question"],
        model=model,
        index=index,
        chunks=chunks,
        top_k=3,
    )

    score = reciprocal_rank(
        results,
        item["expected_text"],
    )

    reciprocal_ranks.append(score)

mrr = sum(reciprocal_ranks) / len(
    reciprocal_ranks
)

print(f"MRR@3: {mrr:.3f}")

MRR@3: 1.000


In [27]:
def precision_at_k(
    retrieved_results,
    expected_text,
):
    """
    Calculate Precision@K.

    Relevant retrieved chunks / total retrieved chunks.
    """

    if not retrieved_results:
        return 0.0

    relevant = 0

    for result in retrieved_results:

        if (
            expected_text.lower()
            in result["text"].lower()
        ):
            relevant += 1

    return relevant / len(retrieved_results)

In [28]:
item = questions[0]

results = retrieve(
    query=item["question"],
    model=model,
    index=index,
    chunks=chunks,
    top_k=3,
)

precision = precision_at_k(
    results,
    item["expected_text"],
)

print(f"Precision@3: {precision:.2%}")

Precision@3: 100.00%


In [29]:
precision_scores = []

for item in questions:

    results = retrieve(
        query=item["question"],
        model=model,
        index=index,
        chunks=chunks,
        top_k=3,
    )

    precision = precision_at_k(
        results,
        item["expected_text"],
    )

    precision_scores.append(
        precision
    )

average_precision = (
    sum(precision_scores)
    / len(precision_scores)
)

print(
    f"Average Precision@3: "
    f"{average_precision:.2%}"
)

Average Precision@3: 100.00%
